In [1]:
# ── Task 1: Create & Justify Engineered Features ──

import pandas as pd
import numpy as np
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import train_test_split

# ── Load data (same as Day 1) ──
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
columns = ['age','workclass','fnlwgt','education','education_num','marital_status',
           'occupation','relationship','race','sex','capital_gain','capital_loss',
           'hours_per_week','native_country','income']
df = pd.read_csv(url, names=columns, na_values=' ?', skipinitialspace=True)
df.dropna(inplace=True)

# ── Same train/test split as Day 1 (stratified, random_state=42) ──
X = df.drop('income', axis=1)
y = (df['income'] == '>50K').astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Positive rate: {y_train.mean():.4f} (train), {y_test.mean():.4f} (test)')

# ── Create 7 Engineered Features ──

def engineer_features(df):
    """Create engineered features from raw columns. Uses only current-row data (no leakage)."""
    out = pd.DataFrame(index=df.index)

    # 1. has_capital_gain: binary flag
    #    Most people have capital_gain=0. The gap between 0 and any gain is what matters.
    out['has_capital_gain'] = (df['capital_gain'] > 0).astype(int)

    # 2. log_capital_gain: log(1 + capital_gain)
    #    Capital gain is wildly skewed (0 to 99999). Log compresses the range.
    out['log_capital_gain'] = np.log1p(df['capital_gain'])

    # 3. has_capital_loss: binary flag
    #    Same logic as gain — presence/absence matters more than exact value.
    out['has_capital_loss'] = (df['capital_loss'] > 0).astype(int)

    # 4. age_group: life-cycle bins
    #    Income follows a life cycle (low young → peaks mid-career → drops after retirement).
    out['age_group'] = pd.cut(
        df['age'],
        bins=[0, 24, 39, 54, 64, 100],
        labels=['<25', '25-39', '40-54', '55-64', '65+']
    )

    # 5. hours_category: work intensity bins
    #    Part-time vs full-time vs overtime → different income brackets.
    out['hours_category'] = pd.cut(
        df['hours_per_week'],
        bins=[0, 19, 39, 49, 100],
        labels=['<20', '20-39', '40-49', '50+']
    )

    # 6. higher_ed: binary flag (Bachelors or above)
    #    Clear threshold: college degree vs no degree.
    out['higher_ed'] = (df['education_num'] >= 14).astype(int)

    # 7. edu_hours_interaction: education_num × hours_per_week
    #    Combination: highly educated + long hours = highest earners.
    out['edu_hours_interaction'] = df['education_num'] * df['hours_per_week']

    return out

# ── Apply to train and test ──
eng_train = engineer_features(X_train)
eng_test = engineer_features(X_test)

print('\nEngineered features created:')
print(eng_train.columns.tolist())
print(f'Shape: {eng_train.shape}')
display(eng_train.head())

# ── Compute Mutual Information scores ──
# MI measures: how much does knowing this feature reduce uncertainty about income?
# MI = 0 → feature is useless. Higher = more informative.
# Need numeric-only input for MI (age_group and hours_category are categorical)
eng_train_numeric = pd.get_dummies(eng_train, drop_first=True)

mi_scores = mutual_info_classif(eng_train_numeric, y_train, random_state=42)

# Build feature dictionary
feature_dict = pd.DataFrame({
    'Feature': eng_train_numeric.columns,
    'MI_Score': np.round(mi_scores, 4)
}).sort_values('MI_Score', ascending=False).reset_index(drop=True)

print('\nFeature Dictionary — Mutual Information Scores:')
print('(Higher MI = more informative about income >50K)\n')
display(feature_dict)
print(f'\nBest feature: {feature_dict.iloc[0]["Feature"]} (MI={feature_dict.iloc[0]["MI_Score"]})')

Train: (26048, 14), Test: (6513, 14)
Positive rate: 0.2408 (train), 0.2407 (test)

Engineered features created:
['has_capital_gain', 'log_capital_gain', 'has_capital_loss', 'age_group', 'hours_category', 'higher_ed', 'edu_hours_interaction']
Shape: (26048, 7)


,has_capital_gain,log_capital_gain,has_capital_loss,age_group,hours_category,higher_ed,edu_hours_interaction
15738,0,0.000000,0,25-39,40-49,0,585
27985,0,0.000000,0,40-54,40-49,1,630
30673,0,0.000000,0,<25,20-39,0,252
9505,1,8.832004,0,40-54,40-49,0,400
26417,0,0.000000,0,<25,40-49,0,520



Feature Dictionary — Mutual Information Scores:
(Higher MI = more informative about income >50K)



,Feature,MI_Score
0,log_capital_gain,0.0843
1,edu_hours_interaction,0.0829
2,higher_ed,0.0317
3,has_capital_gain,0.0260
4,age_group_40-54,0.0253
5,hours_category_50+,0.0230
6,hours_category_20-39,0.0168
7,has_capital_loss,0.0102
8,age_group_25-39,0.0036
9,age_group_55-64,0.0025



Best feature: log_capital_gain (MI=0.0843)


In [5]:
# ── Task 2: Rebuild Pipeline with Engineered Features ──

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

def engineer_features(df):
    """Create engineered features from raw columns. Returns original + new columns."""
    out = df.copy() 
    # 1. has_capital_gain: binary flag
    out['has_capital_gain'] = (df['capital_gain'] > 0).astype(int)
    # 2. log_capital_gain: log(1 + capital_gain)
    out['log_capital_gain'] = np.log1p(df['capital_gain'])
    # 3. has_capital_loss: binary flag
    out['has_capital_loss'] = (df['capital_loss'] > 0).astype(int)
    # 4. age_group: life-cycle bins
    out['age_group'] = pd.cut(df['age'], bins=[0,24,39,54,64,100], labels=['<25','25-39','40-54','55-64','65+'])
    # 5. hours_category: work intensity bins
    out['hours_category'] = pd.cut(df['hours_per_week'], bins=[0,19,39,49,100], labels=['<20','20-39','40-49','50+'])
    # 6. higher_ed: binary flag
    out['higher_ed'] = (df['education_num'] >= 14).astype(int)
    # 7. edu_hours_interaction: education_num × hours_per_week
    out['edu_hours_interaction'] = df['education_num'] * df['hours_per_week']
    return out

# ── Verify the fix ──
check = engineer_features(X_train)
print(f'Output columns: {check.shape[1]} (should be 21 = 14 original + 7 engineered)')
print(f'Columns: {check.columns.tolist()}')

# ── Define which columns are numeric vs categorical AFTER engineering ──
numeric_cols = [
    'age', 'fnlwgt', 'education_num', 'capital_gain', 'capital_loss',
    'hours_per_week',
    'has_capital_gain', 'log_capital_gain', 'has_capital_loss',
    'higher_ed', 'edu_hours_interaction'
]

categorical_cols = [
    'workclass', 'education', 'marital_status', 'occupation',
    'relationship', 'race', 'sex', 'native_country',
    'age_group', 'hours_category'
]

# ── Build numeric sub-pipeline ──
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# ── Build categorical sub-pipeline ──
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# ── Combine into ColumnTransformer ──
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numeric_cols),
        ('cat', categorical_pipeline, categorical_cols)
    ],
    remainder='drop'
)

# ── Wrap engineer_features in FunctionTransformer ──
engineer = FunctionTransformer(engineer_features)

# ── Build full pipeline (engineer → preprocess → model) ──
full_pipeline = Pipeline([
    ('engineer', engineer),
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        solver='liblinear',
        random_state=42,
        max_iter=1000
    ))
])

# ── Fit on training data only ──
full_pipeline.fit(X_train, y_train)

# ── Evaluate on test data ──
y_pred = full_pipeline.predict(X_test)
y_proba = full_pipeline.predict_proba(X_test)[:, 1]

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

print('\nFull Pipeline (Engineer + Preprocess + LR) — Test Set Results:')
print(f'  Precision:  {precision:.4f}')
print(f'  Recall:     {recall:.4f}')
print(f'  F1:         {f1:.4f}')
print(f'  ROC AUC:    {roc_auc:.4f}')

# ── Check feature count ──
feature_names = full_pipeline.named_steps['preprocessor'].get_feature_names_out()
print(f'\nTotal features after preprocessing: {len(feature_names)}')
print(f'Original features: 14 → Engineered: 21 → After one-hot: {len(feature_names)}')

Output columns: 21 (should be 21 = 14 original + 7 engineered)
Columns: ['age', 'workclass', 'fnlwgt', 'education', 'education_num', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'has_capital_gain', 'log_capital_gain', 'has_capital_loss', 'age_group', 'hours_category', 'higher_ed', 'edu_hours_interaction']

Full Pipeline (Engineer + Preprocess + LR) — Test Set Results:
  Precision:  0.7528
  Recall:     0.6352
  F1:         0.6890
  ROC AUC:    0.9181

Total features after preprocessing: 122
Original features: 14 → Engineered: 21 → After one-hot: 122


In [ ]:
# ── Task 3: Cross-Validated Model Comparison (5-Fold Stratified) ──

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ── Define CV strategy ──
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ── Define three pipelines (same engineer + preprocessor, different classifier) ──
pipelines = {
    'Logistic Regression': Pipeline([
        ('engineer', FunctionTransformer(engineer_features)),
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(solver='liblinear', random_state=42, max_iter=1000))
    ]),
    'Random Forest': Pipeline([
        ('engineer', FunctionTransformer(engineer_features)),
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
    ]),
    'HistGradientBoosting': Pipeline([
        ('engineer', FunctionTransformer(engineer_features)),
        ('preprocessor', preprocessor),
        ('classifier', HistGradientBoostingClassifier(random_state=42))
    ])
}

# ── Run cross-validation for each model ──
scoring = ['precision', 'recall', 'f1', 'roc_auc']
results = {}

for name, pipeline in pipelines.items():
    print(f'Training: {name}...')
    cv_results = cross_validate(
        pipeline, X_train, y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=False
    )
    results[name] = cv_results
    print(f'  Done.')
    for metric in scoring:
        scores = cv_results[f'test_{metric}']
        print(f'  {metric:>12s}: {scores.mean():.4f} ± {scores.std():.4f}')
    print()

# ── Build comparison table ──
summary = []
for name in results:
    row = {'Model': name}
    for metric in scoring:
        scores = results[name][f'test_{metric}']
        row[f'{metric}_mean'] = scores.mean()
        row[f'{metric}_std'] = scores.std()
    summary.append(row)

summary_df = pd.DataFrame(summary)
print('Cross-Validation Summary:')
display(summary_df.round(4))

# ── Boxplot visualization ──
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for idx, metric in enumerate(scoring):
    ax = axes[idx]
    data = [results[name][f'test_{metric}'] for name in results]
    ax.boxplot(data, labels=[n[:12] for n in results.keys()])
    ax.set_title(metric.replace('_', ' ').title())
    ax.set_ylabel('Score')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Cross-Validation Fold Scores — Model Comparison', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# ── Print raw fold scores for inspection ──
print('\nRaw fold scores (Precision):')
for name in results:
    scores = results[name]['test_precision']
    print(f'  {name}: {[f"{s:.4f}" for s in scores]}')

Training: Logistic Regression...
  Done.
     precision: 0.7455 ± 0.0083
        recall: 0.6184 ± 0.0108
            f1: 0.6760 ± 0.0093
       roc_auc: 0.9130 ± 0.0045

Training: Random Forest...
  Done.
     precision: 0.7274 ± 0.0091
        recall: 0.6168 ± 0.0072
            f1: 0.6675 ± 0.0046
       roc_auc: 0.9021 ± 0.0043

Training: HistGradientBoosting...
